# Laya V2.6 — Layer 15 Full-Attention Replacement Tournament

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V26_FullAttention_Tournament_Colab.ipynb)

V2.6 is deliberately narrow: it **does not replace sliding attention** and it does not try to convert all full-attention blocks at once. It isolates **ModernBERT full-attention layer 15** to compare all six candidate families under the same train/dev data and step budget.

For layer 15 the notebook trains multiple replacement families against the **unchanged teacher**, screens all candidates on identical cached teacher attention I/O, fully refines all six, then inserts each finalist into an otherwise untouched student and measures end-to-end teacher agreement.

Candidates:

- **V2.5 MultiscaleCeNN** — current baseline.
- **Large MultiscaleCeNN** — tests whether the V2.5 failure is just insufficient CeNN capacity.
- **Integrated Memory V2.2 / 128 features** — learned query-dependent global linear memory + CeNN-style local route + direct value.
- **MemoryFusion V3** — sparse symmetric local QKV + Hedgehog global memory + direct value; recurrent GDN2 is enabled only when the screen result is already close.
- **PDelta3 global** — GDN2 + learned bidirectional global linear memory + local value route.
- **PDelta3 Local32** — same as PDelta3 global plus exact query-dependent Local32 attention; this is a hybrid and is included as a quality upper bound.

The official test split is **not used for architecture selection**. V2.6 selects winners only from the train/dev split.


In [1]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
#@title 2. Imports, model, and tournament configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.laya_lab.integrated import IntegratedMemoryV22Attention
from tinycenn_lm.laya_lab.memory_fusion_v3 import MemoryFusionV3Attention
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import (
    require_finite, guarded_step, forward_with_markers,
    marker_alignment_loss, decision_distill_loss,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
TARGET_LAYERS = (15,)  # Isolate layer 15 for a fair architecture comparison

TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}

# Two-stage tournament: all candidates screen quickly; only the best two per
# layer receive the full local refinement and end-to-end recovery.
SCREEN_STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1600 #@param {type:"integer"}
E2E_STEPS = 200 #@param {type:"integer"}
TOP_K = 6 #@param {type:"integer"}  # Refine all six candidate families
TRAIN_CACHE_BATCHES = 48 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
E2E_CHECK_EVERY = 25 #@param {type:"integer"}
DEV_GATE_LIMIT = 128 #@param {type:"integer"}

# Final single-layer quality gates. A winner is still reported when nobody
# passes, but it is explicitly marked as diagnostic-only.
MAX_LOCAL_NMSE = 0.20
MIN_LOCAL_COSINE = 0.90
MIN_TEACHER_AGREEMENT = 0.95
MAX_MEAN_JS = 0.02

OUTPUT_DIR = Path("/content/Laya_V26_Layer15_Tournament")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("V2.6 tournament requires a CUDA GPU.")

device = torch.device("cuda")
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported(including_emulation=False)
    else torch.float16
)

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=[
        "rl_agent_config.json",
        "model.safetensors",
        "encoder/*",
        "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))

teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert all(i in full_layers for i in TARGET_LAYERS), (TARGET_LAYERS, full_layers)

print("device:", torch.cuda.get_device_name(0))
print("amp:", amp_dtype)
print("source:", SOURCE_MODEL)
print("full-attention layers:", full_layers)
print("V2.6 target layers:", TARGET_LAYERS)
print("screen/refine/e2e:", SCREEN_STEPS, REFINE_STEPS, E2E_STEPS)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

rl_agent_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

device: Tesla T4
amp: torch.float16
source: convaiinnovations/laya-typed-decisions
full-attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]
V2.6 target layers: (15,)
screen/refine/e2e: 400 1600 200


In [3]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_embeddings=entry["pos"],
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V2.6 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
print("official test remains untouched")


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

train/dev decisions: 3000 500
official test remains untouched


In [4]:
#@title 4. Candidate registry
CANDIDATE_NAMES = (
    "cenn_v25",
    "cenn_large",
    "integrated_v22_128",
    "memory_fusion_v3",
    "pdelta3_global",
    "pdelta3_local32",
)

CANDIDATE_INFO = {
    "cenn_v25": {
        "label": "V2.5 MultiscaleCeNN 256x2",
        "hybrid_attention": False,
        "screen_lr": (1.0e-2, 4.0e-3),
        "refine_lr": (4.0e-3, 2.0e-3),
        "e2e_lr": 3.0e-4,
    },
    "cenn_large": {
        "label": "Large MultiscaleCeNN 512x4",
        "hybrid_attention": False,
        "screen_lr": (8.0e-3, 3.0e-3),
        "refine_lr": (3.0e-3, 1.0e-3),
        "e2e_lr": 2.0e-4,
    },
    "integrated_v22_128": {
        "label": "Integrated Memory V2.2 F=128",
        "hybrid_attention": False,
        "screen_lr": (1.0e-2, 3.0e-3),
        "refine_lr": (3.0e-3, 5.0e-4),
        "e2e_lr": 3.0e-4,
    },
    "memory_fusion_v3": {
        "label": "MemoryFusion V3 F=96 R=64",
        "hybrid_attention": False,
        "screen_lr": (1.0e-3, 6.0e-4),
        "refine_lr": (6.0e-4, 2.0e-4),
        "e2e_lr": 1.0e-4,
    },
    "pdelta3_global": {
        "label": "PDelta3 global F=96",
        "hybrid_attention": False,
        "screen_lr": (1.0e-2, 3.0e-3),
        "refine_lr": (3.0e-3, 1.0e-3),
        "e2e_lr": 2.0e-4,
    },
    "pdelta3_local32": {
        "label": "PDelta3 global + Local32",
        "hybrid_attention": True,
        "screen_lr": (1.0e-2, 3.0e-3),
        "refine_lr": (3.0e-3, 1.0e-3),
        "e2e_lr": 2.0e-4,
    },
}

def build_candidate(name, original):
    if name == "cenn_v25":
        repl = MultiscaleCeNN(
            original,
            cell_dim=256,
            steps=2,
            scales=(1, 4, 16),
        )
    elif name == "cenn_large":
        repl = MultiscaleCeNN(
            original,
            cell_dim=512,
            steps=4,
            scales=(1, 2, 4, 8, 16, 32),
        )
    elif name == "integrated_v22_128":
        repl = IntegratedMemoryV22Attention(
            original,
            feature_dim=128,
            local_kernel=5,
        )
    elif name == "memory_fusion_v3":
        repl = MemoryFusionV3Attention(
            original,
            feature_dim=96,
            memory_rank=64,
            dilations=(1, 2, 4, 8, 16, 32, 64),
        )
    elif name in ("pdelta3_global", "pdelta3_local32"):
        repl = PDelta3GDN2CLVRAttention(
            original,
            feature_dim=96,
            conv_kernel=4,
            chunk_size=32,
            local_kernel=5,
            local_window=(32 if name == "pdelta3_local32" else 0),
            local_gate_init=0.72,
        )
        # Match the optimized PDelta recipe: QKV frozen, low-LR Wo calibration.
        repl.Wo.float()
        for p in repl.Wo.parameters():
            p.requires_grad = True
    else:
        raise KeyError(name)
    return repl.to(device)

def _unique_params(params):
    seen, out = set(), []
    for p in params:
        if p.requires_grad and id(p) not in seen:
            seen.add(id(p))
            out.append(p)
    return out

def candidate_optimizer(name, repl, phase):
    start, end = CANDIDATE_INFO[name][f"{phase}_lr"]
    groups = []

    if isinstance(repl, MemoryFusionV3Attention):
        specs = [
            (repl.fast_parameters(), start, end),
            (repl.slow_core_parameters(), start * 0.5, end * 0.5),
            (repl.output_parameters(), start * 0.05, end * 0.05),
        ]
    elif isinstance(repl, PDelta3GDN2CLVRAttention):
        core, out = [], []
        for param_name, p in repl.named_parameters():
            if not p.requires_grad:
                continue
            if param_name.startswith("Wo."):
                out.append(p)
            elif not param_name.startswith("Wqkv.") and not param_name.startswith("out_drop."):
                core.append(p)
        specs = [
            (core, start, end),
            (out, start * 0.10, end * 0.10),
        ]
    else:
        params = (
            repl.trainable_core_parameters()
            if hasattr(repl, "trainable_core_parameters")
            else [p for p in repl.parameters() if p.requires_grad]
        )
        specs = [(params, start, end)]

    param_groups = []
    clip_params = []
    for params, s, e in specs:
        params = _unique_params(params)
        if not params:
            continue
        param_groups.append({
            "params": params,
            "lr": float(s),
            "_start_lr": float(s),
            "_end_lr": float(e),
        })
        clip_params.extend(params)

    if not param_groups:
        raise RuntimeError(f"{name}: no trainable parameters")

    opt = torch.optim.AdamW(
        param_groups,
        betas=(0.9, 0.95),
        weight_decay=2e-4,
    )
    return opt, _unique_params(clip_params)

def cosine_value(step, total, start, end):
    if total <= 1:
        return float(end)
    x = min(1.0, max(0.0, step / float(total - 1)))
    return float(end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * x)))

print("Candidates:")
for name in CANDIDATE_NAMES:
    print(" ", name, "->", CANDIDATE_INFO[name]["label"])


Candidates:
  cenn_v25 -> V2.5 MultiscaleCeNN 256x2
  cenn_large -> Large MultiscaleCeNN 512x4
  integrated_v22_128 -> Integrated Memory V2.2 F=128
  memory_fusion_v3 -> MemoryFusion V3 F=96 R=64
  pdelta3_global -> PDelta3 global F=96
  pdelta3_local32 -> PDelta3 global + Local32


In [7]:
#@title 5. Candidate forward/backward preflight
# Fail early before spending thousands of optimizer steps. This checks every
# candidate on real teacher hidden states, the actual CUDA autocast dtype, and
# the ModernBERT full-attention mask/position-embedding signature.
preflight_layer = TARGET_LAYERS[0]
preflight_batch = batch(dev_items, 0, min(BATCH, 2))
preflight_target = attn_io(teacher, preflight_layer, preflight_batch)

for name in CANDIDATE_NAMES:
    repl = build_candidate(
        name,
        teacher.encoder.layers[preflight_layer].attn,
    )
    repl.train()
    if hasattr(repl, "out_drop"):
        repl.out_drop.eval()

    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        pred, _ = repl(
            preflight_target["x"],
            position_embeddings=preflight_target["pos"],
            attention_mask=preflight_target["mask"],
        )
        valid = preflight_batch["attention_mask"].bool()
        m = valid[:, :, None].float()
        loss = (
            ((pred.float() - preflight_target["y"].float()) ** 2) * m
        ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)

    require_finite(pred, name + " preflight output")
    require_finite(loss, name + " preflight loss")

    params = [p for p in repl.parameters() if p.requires_grad]
    grads = torch.autograd.grad(
        loss,
        params,
        allow_unused=True,
    )
    finite_grads = [
        g for g in grads
        if g is not None and torch.isfinite(g).all()
    ]
    if not finite_grads or sum(float(g.abs().sum()) for g in finite_grads) <= 0:
        raise RuntimeError(name + ": no finite non-zero preflight gradients")

    print(
        f"{name:<22} PASS "
        f"loss={float(loss.detach()):.5f} "
        f"trainable={sum(p.numel() for p in params):,}"
    )
    del repl, pred, loss, grads, finite_grads
    torch.cuda.empty_cache()

del preflight_batch, preflight_target


cenn_v25               PASS loss=1.57556 trainable=592,132
cenn_large             PASS loss=1.59104 trainable=1,315,335
integrated_v22_128     PASS loss=1.25682 trainable=267,344
memory_fusion_v3       PASS loss=1.96093 trainable=1,788,800
pdelta3_global         PASS loss=1.22091 trainable=1,861,728
pdelta3_local32        PASS loss=1.26171 trainable=1,862,768


In [8]:
#@title 6. Local teacher-grounded training
def state_cpu(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def fit_local(name, repl, train_cache, probe_cache, steps, phase):
    opt, clip_params = candidate_optimizer(name, repl, phase)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_dtype == torch.float16,
        init_scale=128.0,
    )
    best = local_metrics_from_cache(repl, probe_cache)
    best_state = state_cpu(repl)
    best_step = 0
    updates = attempts = 0
    max_attempts = int(steps) + 256

    print(
        f"  {phase} start: NMSE={best['nmse']:.4f} "
        f"cos={best['cosine']:.4f}"
    )

    while updates < steps:
        if attempts >= max_attempts:
            raise RuntimeError(
                f"{name}: too many skipped AMP updates "
                f"({updates}/{steps} successful)"
            )

        for group in opt.param_groups:
            group["lr"] = cosine_value(
                updates,
                steps,
                group["_start_lr"],
                group["_end_lr"],
            )

        entry = train_cache[attempts % len(train_cache)]
        repl.train()
        if hasattr(repl, "out_drop"):
            repl.out_drop.eval()
        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                entry["x"],
                position_embeddings=entry["pos"],
                attention_mask=entry["mask"],
            )
            valid = entry["valid"]
            m = valid[:, :, None].float()
            mse = (
                ((pred.float() - entry["y"].float()) ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = (
                (entry["y"].float() ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cosine = F.cosine_similarity(
                pred.float()[valid],
                entry["y"].float()[valid],
                dim=-1,
            ).mean()
            loss = nmse + 1.10 * (1.0 - cosine)

        updated = guarded_step(
            loss,
            opt,
            scaler,
            [(clip_params, 1.0)],
            context=f"V2.6 {name} {phase} update {updates + 1}",
        )
        attempts += 1
        if not updated:
            continue

        updates += 1
        if (
            updates == 1
            or updates % CHECK_EVERY == 0
            or updates == steps
        ):
            probe = local_metrics_from_cache(repl, probe_cache)
            if probe["score"] < best["score"]:
                best = probe
                best_state = state_cpu(repl)
                best_step = updates
            print(
                f"    {updates:04d}/{steps} "
                f"NMSE={probe['nmse']:.4f} cos={probe['cosine']:.4f} "
                f"best={best['score']:.4f}"
            )

    repl.load_state_dict(best_state, strict=True)
    repl.eval()
    return {
        "phase": phase,
        "steps": int(steps),
        "attempts": int(attempts),
        "best_step": int(best_step),
        "local": best,
    }, best_state

def maybe_enable_memory(name, repl, screen_local):
    if (
        name == "memory_fusion_v3"
        and isinstance(repl, MemoryFusionV3Attention)
        and not repl.memory_enabled
        and screen_local["nmse"] <= 0.36
        and screen_local["cosine"] >= 0.80
    ):
        repl.enable_memory_refinement()
        return True
    return False


In [9]:
#@title 7. End-to-end teacher agreement and short finalist recovery
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []

    for off in range(0, limit, BATCH):
        selected = items[off:min(off + BATCH, limit)]
        b = direct_batch(selected)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "student validation logits")

        mask = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~mask, -1e9)
        sm = sl.float().masked_fill(~mask, -1e9)
        tp = torch.softmax(tm, -1)
        sp = torch.softmax(sm, -1)
        tpred = tm.argmax(-1)
        spred = sm.argmax(-1)

        gold = torch.tensor(
            [item["gold_index"] for item in selected],
            device=device,
            dtype=torch.long,
        )
        agree += int((tpred == spred).sum())
        teacher_gold += int((tpred == gold).sum())
        student_gold += int((spred == gold).sum())
        total += len(selected)

        mid = 0.5 * (tp + sp)
        j = 0.5 * (
            tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        j += 0.5 * (
            sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        js.extend(j.cpu().tolist())

    return {
        "decisions": int(total),
        "agreement": agree / max(1, total),
        "mean_js": float(np.mean(js)),
        "teacher_gold_accuracy": teacher_gold / max(1, total),
        "student_gold_accuracy": student_gold / max(1, total),
        "accuracy_gap": (student_gold - teacher_gold) / max(1, total),
    }

def agreement_score(metrics):
    return metrics["mean_js"] + 0.50 * (1.0 - metrics["agreement"])

def candidate_trainable_params(repl):
    return _unique_params([p for p in repl.parameters() if p.requires_grad])

def e2e_refine(layer_idx, name, repl, steps=E2E_STEPS):
    original = student_eval.encoder.layers[layer_idx].attn
    student_eval.encoder.layers[layer_idx].attn = repl
    student_eval.eval().requires_grad_(False)

    # requires_grad_(False) above also freezes the inserted candidate; re-enable
    # only the parameters that the candidate recipe is allowed to tune.
    if isinstance(repl, MultiscaleCeNN):
        for p in repl.parameters():
            p.requires_grad = True
    elif isinstance(repl, MemoryFusionV3Attention):
        for p in repl.fast_parameters() + repl.slow_core_parameters() + repl.output_parameters():
            p.requires_grad = True
        for p in repl.Wqkv.parameters():
            p.requires_grad = False
    elif isinstance(repl, PDelta3GDN2CLVRAttention):
        for n, p in repl.named_parameters():
            p.requires_grad = (
                not n.startswith("Wqkv.")
                and not n.startswith("out_drop.")
            )
    else:
        for p in repl.trainable_core_parameters():
            p.requires_grad = True

    params = candidate_trainable_params(repl)
    if not params:
        raise RuntimeError(f"{name}: no end-to-end trainable parameters")

    lr = float(CANDIDATE_INFO[name]["e2e_lr"])
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=1e-4)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_dtype == torch.float16,
        init_scale=128.0,
    )

    baseline = validate_pair(student_eval, dev_items, DEV_GATE_LIMIT)
    best_metrics = baseline
    best_score = agreement_score(baseline)
    best_state = state_cpu(repl)
    best_step = 0
    updates = attempts = 0

    print(
        f"  E2E baseline: agreement={baseline['agreement']:.4f} "
        f"JS={baseline['mean_js']:.5f}"
    )

    try:
        while updates < steps:
            progress = updates / max(1, steps - 1)
            current_lr = lr * (0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress)))
            for group in opt.param_groups:
                group["lr"] = current_lr

            b = batch(train_items, attempts * BATCH + layer_idx * 37, BATCH)
            opt.zero_grad(set_to_none=True)

            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
                tl, ta, teacher_features = forward_with_markers(teacher, b)

            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                sl, sa, student_features = forward_with_markers(student_eval, b)
                loss = decision_distill_loss(
                    sl,
                    tl,
                    b["marker_mask"].bool(),
                    sa,
                    ta,
                    temperature=1.25,
                    rank_margin=0.15,
                    rank_weight=0.15,
                )
                loss = loss + 0.05 * marker_alignment_loss(
                    student_features,
                    teacher_features,
                    b["marker_mask"].bool(),
                )

            updated = guarded_step(
                loss,
                opt,
                scaler,
                [(params, 0.75)],
                context=f"V2.6 {name} layer {layer_idx} E2E {updates + 1}",
            )
            attempts += 1
            if not updated:
                continue
            updates += 1

            if (
                updates == 1
                or updates % E2E_CHECK_EVERY == 0
                or updates == steps
            ):
                metrics = validate_pair(
                    student_eval,
                    dev_items,
                    DEV_GATE_LIMIT,
                )
                score = agreement_score(metrics)
                if score < best_score:
                    best_score = score
                    best_metrics = metrics
                    best_state = state_cpu(repl)
                    best_step = updates
                print(
                    f"    E2E {updates:03d}/{steps} "
                    f"agree={metrics['agreement']:.4f} "
                    f"JS={metrics['mean_js']:.5f} "
                    f"best={best_metrics['agreement']:.4f}"
                )
    finally:
        repl.load_state_dict(best_state, strict=True)
        repl.eval()
        student_eval.encoder.layers[layer_idx].attn = original
        student_eval.eval().requires_grad_(False)

    return {
        "baseline": baseline,
        "selected": best_metrics,
        "best_step": int(best_step),
        "updates": int(updates),
        "attempts": int(attempts),
    }, best_state

@torch.no_grad()
def evaluate_candidate(layer_idx, repl, probe_cache):
    original = student_eval.encoder.layers[layer_idx].attn
    student_eval.encoder.layers[layer_idx].attn = repl
    student_eval.eval()
    try:
        local = local_metrics_from_cache(repl, probe_cache)
        quality = validate_pair(student_eval, dev_items, len(dev_items))

        probe_b = direct_batch(dev_items[:BATCH])
        def timed(model, warmup=5, repeats=20):
            for _ in range(warmup):
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    model(**probe_b)
            torch.cuda.synchronize()
            values = []
            for _ in range(repeats):
                t0 = time.perf_counter()
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    model(**probe_b)
                torch.cuda.synchronize()
                values.append((time.perf_counter() - t0) * 1000.0)
            return float(np.median(values))

        teacher_ms = timed(teacher)
        student_ms = timed(student_eval)
        speedup = teacher_ms / max(student_ms, 1e-9)
    finally:
        student_eval.encoder.layers[layer_idx].attn = original
        student_eval.eval()

    strict = (
        local["nmse"] <= MAX_LOCAL_NMSE
        and local["cosine"] >= MIN_LOCAL_COSINE
        and quality["agreement"] >= MIN_TEACHER_AGREEMENT
        and quality["mean_js"] <= MAX_MEAN_JS
    )
    return {
        "local": local,
        "quality": quality,
        "latency": {
            "teacher_median_ms": teacher_ms,
            "student_median_ms": student_ms,
            "speedup": speedup,
        },
        "strict_pass": bool(strict),
    }


In [ ]:
#@title 8. Run the V2.6 tournament
results = []
winners = {}
screening = {}

for layer_idx in TARGET_LAYERS:
    print("\n" + "=" * 100)
    print(f"V2.6 HARD LAYER {layer_idx}")
    print("=" * 100)

    # Build teacher I/O only for this layer, then release it before moving on.
    train_cache = cache_teacher_io(
        layer_idx,
        train_items,
        TRAIN_CACHE_BATCHES,
        offset=layer_idx * 13,
    )
    probe_cache = cache_teacher_io(
        layer_idx,
        dev_items,
        PROBE_CACHE_BATCHES,
        offset=layer_idx * 7,
    )

    layer_screen = []
    for candidate_idx, name in enumerate(CANDIDATE_NAMES):
        seed = SEED + layer_idx * 100 + candidate_idx
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        print(f"\n--- SCREEN {name} @ layer {layer_idx} ---")
        original = teacher.encoder.layers[layer_idx].attn
        repl = build_candidate(name, original)
        screen_report, screen_state = fit_local(
            name,
            repl,
            train_cache,
            probe_cache,
            SCREEN_STEPS,
            "screen",
        )
        layer_screen.append({
            "name": name,
            "report": screen_report,
            "state": screen_state,
        })
        print(
            f"  SCREEN RESULT {name}: "
            f"NMSE={screen_report['local']['nmse']:.4f} "
            f"cos={screen_report['local']['cosine']:.4f}"
        )
        del repl
        torch.cuda.empty_cache()

    # Local teacher fidelity decides who gets the expensive full refinement.
    layer_screen.sort(
        key=lambda r: (
            r["report"]["local"]["score"],
            -r["report"]["local"]["cosine"],
        )
    )
    finalists = layer_screen[:TOP_K]
    screening[str(layer_idx)] = [
        {
            "name": row["name"],
            "local": row["report"]["local"],
            "rank": rank + 1,
        }
        for rank, row in enumerate(layer_screen)
    ]

    print("\nSCREEN RANKING")
    finalist_names = {row["name"] for row in finalists}
    for rank, row in enumerate(layer_screen, 1):
        m = row["report"]["local"]
        mark = " <== FINALIST" if row["name"] in finalist_names else ""
        print(
            f" {rank}. {row['name']:<22} "
            f"NMSE={m['nmse']:.4f} cos={m['cosine']:.4f} "
            f"score={m['score']:.4f}{mark}"
        )

    layer_finalists = []
    for finalist in finalists:
        name = finalist["name"]
        print(f"\n>>> FULL REFINE {name} @ layer {layer_idx}")

        repl = build_candidate(name, teacher.encoder.layers[layer_idx].attn)
        repl.load_state_dict(finalist["state"], strict=True)

        memory_enabled = maybe_enable_memory(
            name,
            repl,
            finalist["report"]["local"],
        )

        refine_report, _ = fit_local(
            name,
            repl,
            train_cache,
            probe_cache,
            REFINE_STEPS,
            "refine",
        )

        e2e_report, _ = e2e_refine(
            layer_idx,
            name,
            repl,
            E2E_STEPS,
        )

        final_eval = evaluate_candidate(
            layer_idx,
            repl,
            probe_cache,
        )

        checkpoint = OUTPUT_DIR / f"layer_{layer_idx}_{name}.pt"
        torch.save({
            "version": "2.6",
            "source_model": SOURCE_MODEL,
            "layer": int(layer_idx),
            "candidate": name,
            "candidate_info": CANDIDATE_INFO[name],
            "memory_enabled": bool(
                getattr(repl, "memory_enabled", False)
            ),
            "state_dict": state_cpu(repl),
            "screen": finalist["report"],
            "refine": refine_report,
            "e2e": e2e_report,
            "final": final_eval,
        }, checkpoint)

        rec = {
            "layer": int(layer_idx),
            "candidate": name,
            "label": CANDIDATE_INFO[name]["label"],
            "hybrid_attention": bool(CANDIDATE_INFO[name]["hybrid_attention"]),
            "memory_enabled": bool(memory_enabled),
            "screen": finalist["report"],
            "refine": refine_report,
            "e2e": e2e_report,
            "final": final_eval,
            "checkpoint": str(checkpoint),
        }
        results.append(rec)
        layer_finalists.append(rec)

        q = final_eval["quality"]
        l = final_eval["local"]
        print(
            f"FINAL {name}: strict={final_eval['strict_pass']} "
            f"agree={q['agreement']:.4f} JS={q['mean_js']:.5f} "
            f"NMSE={l['nmse']:.4f} cos={l['cosine']:.4f} "
            f"speedup={final_eval['latency']['speedup']:.3f}x"
        )

        del repl
        torch.cuda.empty_cache()

    # Quality first: strict pass > teacher agreement > JS > local cosine > NMSE.
    layer_finalists.sort(
        key=lambda r: (
            not r["final"]["strict_pass"],
            -r["final"]["quality"]["agreement"],
            r["final"]["quality"]["mean_js"],
            -r["final"]["local"]["cosine"],
            r["final"]["local"]["nmse"],
        )
    )
    winners[str(layer_idx)] = layer_finalists[0]

    win = layer_finalists[0]
    print(
        f"\n🏆 LAYER {layer_idx} WINNER: {win['candidate']} | "
        f"strict={win['final']['strict_pass']} | "
        f"agreement={win['final']['quality']['agreement']:.4f} | "
        f"cos={win['final']['local']['cosine']:.4f}"
    )

    del train_cache, probe_cache, layer_screen, layer_finalists
    torch.cuda.empty_cache()



V2.6 HARD LAYER 15

--- SCREEN cenn_v25 @ layer 15 ---
  screen start: NMSE=1.1226 cos=0.0679
    0001/400 NMSE=1.2499 cos=0.3173 best=2.0009
    0100/400 NMSE=0.4659 cos=0.7354 best=0.7570
    0200/400 NMSE=0.4221 cos=0.7629 best=0.6829
    0300/400 NMSE=0.4003 cos=0.7765 best=0.6461
    0400/400 NMSE=0.3900 cos=0.7825 best=0.6293
  SCREEN RESULT cenn_v25: NMSE=0.3900 cos=0.7825

--- SCREEN cenn_large @ layer 15 ---
  screen start: NMSE=1.1409 cos=0.1485
    0001/400 NMSE=2.0095 cos=0.3432 best=2.0775
    0100/400 NMSE=0.4737 cos=0.7312 best=0.7693
    0200/400 NMSE=0.4192 cos=0.7652 best=0.6775
    0300/400 NMSE=0.3943 cos=0.7806 best=0.6356
    0400/400 NMSE=0.3811 cos=0.7887 best=0.6136
  SCREEN RESULT cenn_large: NMSE=0.3811 cos=0.7887

--- SCREEN integrated_v22_128 @ layer 15 ---
  screen start: NMSE=0.9025 cos=0.4181
    0001/400 NMSE=0.8851 cos=0.4281 best=1.5141
    0100/400 NMSE=0.3708 cos=0.7941 best=0.5973
    0200/400 NMSE=0.3002 cos=0.8368 best=0.4798
    0300/400 NMSE=0

In [ ]:
#@title 9. Summary and saved V2.6 report
summary = []
for layer_idx in TARGET_LAYERS:
    win = winners[str(layer_idx)]
    final = win["final"]
    summary.append({
        "layer": int(layer_idx),
        "winner": win["candidate"],
        "label": win["label"],
        "strict_pass": bool(final["strict_pass"]),
        "hybrid_attention": bool(win["hybrid_attention"]),
        "local_nmse": float(final["local"]["nmse"]),
        "local_cosine": float(final["local"]["cosine"]),
        "teacher_agreement": float(final["quality"]["agreement"]),
        "mean_js": float(final["quality"]["mean_js"]),
        "teacher_gold_accuracy": float(final["quality"]["teacher_gold_accuracy"]),
        "student_gold_accuracy": float(final["quality"]["student_gold_accuracy"]),
        "accuracy_gap": float(final["quality"]["accuracy_gap"]),
        "speedup": float(final["latency"]["speedup"]),
        "checkpoint": win["checkpoint"],
    })

print("\n" + "=" * 112)
print("V2.6 PER-LAYER WINNERS")
print("=" * 112)
print(
    f"{'layer':>5}  {'winner':<22} {'pass':>5} {'NMSE':>8} {'cos':>8} "
    f"{'agree':>8} {'JS':>9} {'speed':>8} {'hybrid':>7}"
)
for row in summary:
    print(
        f"{row['layer']:>5}  {row['winner']:<22} "
        f"{str(row['strict_pass']):>5} "
        f"{row['local_nmse']:>8.4f} "
        f"{row['local_cosine']:>8.4f} "
        f"{row['teacher_agreement']:>8.4f} "
        f"{row['mean_js']:>9.5f} "
        f"{row['speedup']:>7.3f}x "
        f"{str(row['hybrid_attention']):>7}"
    )

report = {
    "version": "2.6",
    "source_model": SOURCE_MODEL,
    "target_layers": list(TARGET_LAYERS),
    "selection_policy": (
        "single-layer isolated tournament; all other teacher attention remains native; "
        "strict-pass first, then teacher agreement, JS, local cosine, NMSE"
    ),
    "screen_steps": SCREEN_STEPS,
    "refine_steps": REFINE_STEPS,
    "e2e_steps": E2E_STEPS,
    "top_k": TOP_K,
    "candidates": CANDIDATE_INFO,
    "screening": screening,
    "finalists": results,
    "winners": summary,
    "official_test_used": False,
}
report_path = OUTPUT_DIR / "v26_tournament_report.json"
report_path.write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8",
)
print("\nSaved:", report_path)

failed = [r for r in summary if not r["strict_pass"]]
if failed:
    print("\nDiagnostic-only winners (did not pass all strict gates):")
    for row in failed:
        print(
            f"  layer {row['layer']}: {row['winner']} "
            f"agreement={row['teacher_agreement']:.4f} "
            f"cos={row['local_cosine']:.4f}"
        )
else:
    print("\nLayer 15 has a strict-passing replacement.")

print("\nLAYER 15 FULL FINALIST RANKING")
ranked = sorted([r for r in results if r["layer"] == 15], key=lambda r: (not r["final"]["strict_pass"], -r["final"]["quality"]["agreement"], r["final"]["quality"]["mean_js"], -r["final"]["local"]["cosine"], r["final"]["local"]["nmse"]))
for rank, row in enumerate(ranked, 1):
    q, loc, lat = row["final"]["quality"], row["final"]["local"], row["final"]["latency"]
    print(f"{rank}. {row['candidate']:<22} pass={row['final']['strict_pass']} agree={q['agreement']:.4f} JS={q['mean_js']:.5f} NMSE={loc['nmse']:.4f} cosine={loc['cosine']:.4f} speedup={lat['speedup']:.3f}x")


In [ ]:
#@title 10. Optional: test winner insertion (disabled by default)
# The tournament above compares replacements in isolation so architecture quality
# is not confounded by upstream approximation error. After you inspect the winners,
# set this to True to test whether the winners also compose safely.
RUN_COMBINED_WINNERS = False #@param {type:"boolean"}

if RUN_COMBINED_WINNERS:
    combined, _ = build_checkpoint_model(source_dir)
    combined.to(device).eval().requires_grad_(False)
    combined.encoder.config.reference_compile = False

    installed = []
    for layer_idx in TARGET_LAYERS:
        win = winners[str(layer_idx)]
        name = win["candidate"]
        checkpoint = torch.load(win["checkpoint"], map_location="cpu")
        repl = build_candidate(name, combined.encoder.layers[layer_idx].attn)
        repl.load_state_dict(checkpoint["state_dict"], strict=True)
        repl.eval().requires_grad_(False)
        combined.encoder.layers[layer_idx].attn = repl
        installed.append((layer_idx, name))

        metrics = validate_pair(combined, dev_items, len(dev_items))
        print(
            f"after layer {layer_idx:02d} {name:<22} "
            f"agreement={metrics['agreement']:.4f} "
            f"JS={metrics['mean_js']:.5f}"
        )

    combined_metrics = validate_pair(combined, dev_items, len(dev_items))
    print("\ncombined winners:", installed)
    print(json.dumps(combined_metrics, indent=2))

    if combined_metrics["agreement"] < MIN_TEACHER_AGREEMENT:
        print(
            "Combined model fails the 0.95 agreement gate. "
            "Keep the tournament result per-layer; do not treat the stacked "
            "five-layer model as accepted."
        )
